In [4]:
"""
KNN-Based Undersampling of Normal Class (NearMiss)
====================================================================
Purpose: Reduce the Normal (majority) class using KNN-distance-based
undersampling (NearMiss), while leaving all attack classes (DoS,
Probe, R2L, U2R) completely untouched -- those get balanced via
GAN oversampling separately, not here.

This script uses NearMiss version 1, which selects Normal samples
whose average distance to their K CLOSEST attack-class neighbors is
smallest -- i.e. it keeps Normal samples that sit nearest the
attack-class "boundary" region, which are the most informative
samples for a classifier to learn from (rather than keeping random
Normal samples deep in the majority region).

NOTE: NearMiss version 2 was tried first but is NOT memory-safe at
this scale -- it computes distance from every Normal sample to EVERY
attack sample simultaneously (61573 x 56752 matrix = ~26GB), which
crashes on most machines. Version 1 only computes each sample's K
nearest neighbors (K=3 by default here), which is far cheaper and
gives comparable KNN-based selection logic.

Requires: imbalanced-learn (pip install imbalanced-learn)

Author: Vibha J | GAN Implementation Lead
====================================================================
"""

import pandas as pd
import numpy as np
from imblearn.under_sampling import NearMiss

# ============================================================
# CONFIG -- EDIT THESE PATHS
# ============================================================
X_PATH = "./data/X_train.csv"
Y_PATH = "./data/y_train_multiclass.csv"
OUTPUT_X_PATH = "./data/X_train_undersampled_v2.csv"
OUTPUT_Y_PATH = "./data/y_train_multiclass_undersampled_v2.csv"

NORMAL_LABEL = 4          # from attack_category_encoded: 0=DoS 1=Probe 2=R2L 3=U2R 4=Normal
TARGET_NORMAL_COUNT = 42401  # matched to DoS class size
NEARMISS_VERSION = 1      # NearMiss-1: memory-safe, uses K closest neighbors only (not full class)
N_NEIGHBORS = 3            # K value for NearMiss


def main():
    print("Loading data...")
    X = pd.read_csv(X_PATH)
    y = pd.read_csv(Y_PATH)
    label_col = y.columns[0]
    y_series = y[label_col]

    print(f"\nOriginal class distribution:")
    print(y_series.value_counts().sort_index())
    print(f"Total: {len(y_series)}")

    # Separate Normal class from attack classes
    normal_mask = y_series == NORMAL_LABEL
    attack_mask = ~normal_mask

    X_normal = X[normal_mask].reset_index(drop=True)
    y_normal = y_series[normal_mask].reset_index(drop=True)
    X_attacks = X[attack_mask].reset_index(drop=True)
    y_attacks = y_series[attack_mask].reset_index(drop=True)

    print(f"\nNormal class before undersampling: {len(X_normal)}")
    print(f"Attack classes (untouched): {len(X_attacks)}")

    if TARGET_NORMAL_COUNT >= len(X_normal):
        print(f"\nWARNING: TARGET_NORMAL_COUNT ({TARGET_NORMAL_COUNT}) >= current "
              f"Normal count ({len(X_normal)}). Nothing to undersample. Check your config.")
        return

    # NearMiss needs at least 2 classes to compare distances against.
    # We use Normal (majority, to be reduced) vs. ALL attacks combined
    # (as a single "minority" reference class for distance calculation),
    # then only keep the undersampled Normal rows -- attack rows are
    # discarded from this step and re-added untouched afterward.
    X_for_nearmiss = pd.concat([X_normal, X_attacks], axis=0, ignore_index=True)
    y_for_nearmiss = pd.Series(
        [NORMAL_LABEL] * len(X_normal) + [-1] * len(X_attacks)  # -1 = combined attack reference
    )

    sampling_strategy = {NORMAL_LABEL: TARGET_NORMAL_COUNT}

    print(f"\nRunning NearMiss-{NEARMISS_VERSION} undersampling on Normal class "
          f"(target: {TARGET_NORMAL_COUNT})...")
    nm = NearMiss(version=NEARMISS_VERSION, n_neighbors=N_NEIGHBORS,
                  sampling_strategy=sampling_strategy)
    X_resampled, y_resampled = nm.fit_resample(X_for_nearmiss, y_for_nearmiss)

    # Extract only the undersampled Normal rows (drop the -1 reference rows,
    # which are the original untouched attack rows we'll add back separately)
    normal_result_mask = y_resampled == NORMAL_LABEL
    X_normal_undersampled = X_resampled[normal_result_mask].reset_index(drop=True)

    print(f"Normal class after undersampling: {len(X_normal_undersampled)}")

    # Recombine: undersampled Normal + untouched attack classes
    X_final = pd.concat([X_normal_undersampled, X_attacks], axis=0, ignore_index=True)
    y_final = pd.concat([
        pd.Series([NORMAL_LABEL] * len(X_normal_undersampled)),
        y_attacks
    ], axis=0, ignore_index=True)

    # Shuffle so Normal and attack rows are interleaved, not blocked together
    shuffle_idx = np.random.RandomState(42).permutation(len(X_final))
    X_final = X_final.iloc[shuffle_idx].reset_index(drop=True)
    y_final = y_final.iloc[shuffle_idx].reset_index(drop=True)

    print(f"\nFinal class distribution after undersampling:")
    print(y_final.value_counts().sort_index())
    print(f"Total: {len(y_final)}")

    X_final.to_csv(OUTPUT_X_PATH, index=False)
    y_final.to_frame(name=label_col).to_csv(OUTPUT_Y_PATH, index=False)

    print(f"\nSaved -> {OUTPUT_X_PATH}")
    print(f"Saved -> {OUTPUT_Y_PATH}")
    print("\nSanity check: attack class counts should be UNCHANGED from original:")
    print("Original: DoS=42401 Probe=11168 R2L=2985 U2R=198")
    print("Compare against printed distribution above (labels 0,1,2,3).")


if __name__ == "__main__":
    main()

Loading data...

Original class distribution:
attack_category_encoded
0    42401
1    11168
2     2985
3      198
4    61573
Name: count, dtype: int64
Total: 118325

Normal class before undersampling: 61573
Attack classes (untouched): 56752

Running NearMiss-1 undersampling on Normal class (target: 42401)...
Normal class after undersampling: 42401

Final class distribution after undersampling:
0    42401
1    11168
2     2985
3      198
4    42401
Name: count, dtype: int64
Total: 99153

Saved -> ./data/X_train_undersampled_v2.csv
Saved -> ./data/y_train_multiclass_undersampled_v2.csv

Sanity check: attack class counts should be UNCHANGED from original:
Original: DoS=42401 Probe=11168 R2L=2985 U2R=198
Compare against printed distribution above (labels 0,1,2,3).


In [2]:
!pip install imbalanced-learn